In [1]:
library(MLmetrics)
library(dplyr)
library(randomForest)
set.seed(2) 

Warning message:
"le package 'MLmetrics' a été compilé avec la version R 4.2.3"

Attachement du package : 'MLmetrics'


L'objet suivant est masqué depuis 'package:base':

    Recall


Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"

Attachement du package : 'dplyr'


Les objets suivants sont masqués depuis 'package:stats':

    filter, lag


Les objets suivants sont masqués depuis 'package:base':

    intersect, setdiff, setequal, union


randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.


Attachement du package : 'randomForest'


L'objet suivant est masqué depuis 'package:dplyr':

    combine




In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [3]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
datam<-merge(data,data_labels,by=c('building_id','building_id'))



In [4]:
ncol(datam)

[1] 40

We do not need to one hot encode categorical variables for randomForest.

In [7]:
n_trees <- c(10,20,100,200,500)
accuracy_vec <- list()
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))

for (i in n_trees){ 
    for (j in c(3,6,12,24)){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec<-append(accuracy_vec,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',tail(accuracy_vec,n=1)))
    }
}
plot(x=n_trees,y=accuracy_vec,main = "Number of trees influence",xlab = "Nbr of trees",ylab = "F1 Score") 
accuracy_vec


[1] "Number of trees used: 10 mTry: 12"
ntree      OOB      1      2      3
    1:  35.45% 52.84% 29.82% 40.03%
    2:  35.97% 52.16% 29.82% 41.80%
    3:  35.79% 51.32% 29.31% 42.38%
    4:  35.38% 51.22% 28.58% 42.41%
    5:  34.77% 51.10% 27.62% 42.24%
    6:  34.29% 50.82% 26.95% 42.03%
    7:  33.84% 50.97% 26.29% 41.78%
    8:  33.15% 50.49% 25.43% 41.31%
    9:  32.68% 50.42% 24.79% 41.03%
   10:  32.16% 50.17% 24.10% 40.71%
[1] "Best F1 Score -  10 trees - mtry 12 : 0.711977897584467"
[1] "Number of trees used: 10 mTry: 24"
ntree      OOB      1      2      3
    1:  35.42% 52.52% 30.25% 39.44%
    2:  35.38% 51.04% 29.96% 40.21%
    3:  35.10% 50.09% 29.45% 40.48%
    4:  34.74% 49.20% 28.85% 40.65%
    5:  34.37% 49.20% 28.18% 40.71%
    6:  33.88% 48.99% 27.52% 40.42%
    7:  33.41% 49.13% 26.77% 40.22%
    8:  32.96% 48.93% 26.16% 39.97%
    9:  32.49% 48.78% 25.54% 39.66%
   10:  32.06% 48.90% 24.88% 39.48%
[1] "Best F1 Score -  10 trees - mtry 24 : 0.709042420521479"
[1] 

Warning message in randomForest.default(x = train_data[, -c(target_variable)], y = as.factor(train_data[, :
"invalid mtry: reset to within valid range"


: 

: 